# KG Analysis (Phase 2)

Analyze Neo4j KG size, relation distribution, and edge confidence after extraction.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from fair_kg_rag.kg.kg_builder import KnowledgeGraph

In [ ]:
kg_dir = ROOT / 'data' / 'kg'
metadata_files = sorted(kg_dir.glob('*_metadata.json')) if kg_dir.exists() else []
if metadata_files:
    meta_df = pd.DataFrame([pd.read_json(path, typ='series') for path in metadata_files])
    display(meta_df)
else:
    print('No KG metadata snapshot found yet. Run scripts/extract_kg.py first.')

In [ ]:
kg = None
kg_summary = {}
edge_df = pd.DataFrame()

try:
    kg = KnowledgeGraph()
    kg_summary = kg.summary()
    print('KG summary:', kg_summary)

    edges = kg.get_all_edges()
    edge_df = pd.DataFrame(
        [
            {
                'source': src,
                'target': dst,
                'relation': attrs.get('relation'),
                'confidence': attrs.get('confidence'),
                'source_chunk_id': attrs.get('source_chunk_id'),
            }
            for src, dst, attrs in edges
        ]
    )
    print(f'Loaded {len(edge_df)} edges from Neo4j')
except Exception as exc:
    print('Could not connect/query Neo4j KG:', exc)
finally:
    if kg is not None:
        kg.close()

In [ ]:
if not edge_df.empty and 'relation' in edge_df:
    relation_counts = edge_df['relation'].fillna('unknown').value_counts().head(20)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=relation_counts.values, y=relation_counts.index)
    plt.title('Top Relation Types')
    plt.xlabel('Count')
    plt.ylabel('Relation')
    plt.tight_layout()
    plt.show()
else:
    print('No relation data available yet.')

In [ ]:
if not edge_df.empty and 'confidence' in edge_df:
    conf = pd.to_numeric(edge_df['confidence'], errors='coerce').dropna()
    if not conf.empty:
        plt.figure(figsize=(8, 4))
        sns.histplot(conf, bins=30)
        plt.title('Edge Confidence Distribution')
        plt.xlabel('Confidence')
        plt.tight_layout()
        plt.show()
    else:
        print('No numeric confidence values available.')
else:
    print('No confidence data available yet.')